> **Note.**  
> [More Cultural Names](https://github.com/hmlendea/more-cultural-names) (MCN) is used here strictly as an **evaluation oracle**: its attested exonyms serve as a gold standard against which NTE's output is measured. MCN data has not been used to populate, enrich, or tune NTE's own lookup database, which is built independently from [GeoNames](https://www.geonames.org/) alternate-name dumps and [Wikidata](https://www.wikidata.org/) CC0 snapshots. Any agreement between NTE output and MCN entries therefore reflects genuine coverage by NTE's pipeline, not data leakage.
>
> MCN is distributed under the [GPL-3.0 License](https://github.com/hmlendea/more-cultural-names/blob/master/LICENSE.md). This notebook does not redistribute MCN data; it reads the repository in-place from `data/repos/more-cultural-names/`.

In [1]:
import sqlite3
import collections
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from dataclasses import dataclass
from enum import Enum

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

PROJECT_ROOT = Path.cwd().parent
MCN_PATH = PROJECT_ROOT / "data" / "repos" / "more-cultural-names"
MCN_LOCATIONS = MCN_PATH / "locations.xml"
DB_PATH = PROJECT_ROOT / "data" / "names.sqlite"
PACK_ID = "latin"

In [2]:
def get_conn() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def run_query(sql: str, params: tuple = ()) -> pd.DataFrame:
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)

In [3]:
# Get all MCN CK3 names

locations_tree = ET.parse(MCN_LOCATIONS)

ck3_names: dict[tuple[str, int], dict[str, str]] = {}
for loc in locations_tree.iter("LocationEntity"):
    geonames_tag = loc.find("GeoNamesId")
    if geonames_tag is None:
        continue

    geonames_id = int(geonames_tag.text)

    for game_id in loc.findall("GameIds/GameId"):
        if game_id.attrib.get("game") == "CK3":
            title = game_id.text.strip() if game_id.text is not None else ""

            if title.startswith("d_nf") or title.startswith("b_"):
                continue

            ck3_names[(title, geonames_id)] = {
                n.attrib["language"]: n.attrib["value"]
                for n in loc.findall("Names/Name")
            }

In [4]:
# Get all MCN Latin names that have a Geonames tag

ck3_latin_names: dict[tuple[str, int], str] = {}

for loc in ck3_names:
    names = ck3_names[loc]
    if "Latin" in names:
        ck3_latin_names[loc] = names["Latin"]
    else:
        ck3_latin_names[loc] = "MISSING"

In [5]:
ck3_latin_attested = {k: v for k, v in ck3_latin_names.items() if v != "MISSING"}
ck3_latin_missing  = {k for k, v in ck3_latin_names.items() if v == "MISSING"}

In [6]:
total   = len(ck3_latin_names)
attested = len(ck3_latin_attested)
print(f"CK3 titles with GeoNames ID : {total}")
print(f"  — Latin name attested (MCN): {attested} ({attested/total:.1%})")
print(f"  — MISSING                  : {total - attested} ({(total-attested)/total:.1%})")

CK3 titles with GeoNames ID : 218
  — Latin name attested (MCN): 118 (54.1%)
  — MISSING                  : 100 (45.9%)


In [7]:
class MatchStatus(Enum):
    EXACT        = "exact"       # MCN name appears verbatim in DB (any language tag)
    CASE_FOLD    = "case_fold"   # appears after lowercasing
    DB_EMPTY     = "db_empty"    # geonames_id has no alternate names at all
    DB_MISS      = "db_miss"     # alternate names exist, but MCN name not among them

@dataclass
class ComparisonResult:
    title:        str
    geonames_id:  int
    mcn_name:     str
    matched_lang: str | None     # isolanguage of the matching row, if found
    status:       MatchStatus

def compare_one(title: str, geonames_id: int, mcn_name: str) -> ComparisonResult:
    df = run_query("""
        SELECT isolanguage, alternate_name
        FROM alternate_name
        WHERE geonameid = ?
        ORDER BY is_preferred_name DESC, alternate_name
    """, (geonames_id,))

    if df.empty:
        return ComparisonResult(title, geonames_id, mcn_name, None, MatchStatus.DB_EMPTY)

    all_names = df["alternate_name"].tolist()

    # Exact match
    if mcn_name in all_names:
        lang = df.loc[df["alternate_name"] == mcn_name, "isolanguage"].iloc[0]
        return ComparisonResult(title, geonames_id, mcn_name, lang, MatchStatus.EXACT)

    # Case-insensitive match
    lower_map = {n.lower(): (n, row.isolanguage) 
                 for _, row in df.iterrows() 
                 for n in [row.alternate_name]}
    if mcn_name.lower() in lower_map:
        _, lang = lower_map[mcn_name.lower()]
        return ComparisonResult(title, geonames_id, mcn_name, lang, MatchStatus.CASE_FOLD)

    return ComparisonResult(title, geonames_id, mcn_name, None, MatchStatus.DB_MISS)

results = [
    compare_one(title, gid, mcn_name)
    for (title, gid), mcn_name in ck3_latin_attested.items()
]

counts = collections.Counter(r.status for r in results)
for status in MatchStatus:
    n = counts[status]
    print(f"{status.value:15s}  {n:4d}  ({n/len(results):.1%})")

exact              63  (53.4%)
case_fold           0  (0.0%)
db_empty            0  (0.0%)
db_miss            55  (46.6%)


In [8]:
@dataclass(frozen=True)
class NameCandidate:
    source: str          # 'geonames' / 'wikidata'
    source_lang: str     # GeoNames isolanguage, or Wikidata geo_lang/wd_lang
    name: str
    qid: str | None = None

@dataclass
class CombinedComparisonResult:
    title:           str
    geonames_id:     int
    mcn_name:        str
    matched_sources: tuple[str, ...]
    matched_langs:   tuple[str, ...]
    matched_qids:    tuple[str, ...]
    status:          MatchStatus

def _uniq(values):
    return tuple(dict.fromkeys(v for v in values if v is not None))

def get_nte_names_for_geonames_id(geonames_id: int) -> list[NameCandidate]:
    """Return all NTE names for this GeoNames ID from both GeoNames and linked Wikidata QIDs."""
    df = run_query("""
        SELECT
            'geonames' AS source,
            NULL       AS qid,
            isolanguage AS source_lang,
            alternate_name AS name
        FROM alternate_name
        WHERE geonameid = ?

        UNION ALL

        SELECT
            'wikidata' AS source,
            n.qid      AS qid,
            COALESCE(n.geo_lang, n.wd_lang, '') AS source_lang,
            n.name     AS name
        FROM wikidata_location_geonames AS g
        JOIN wikidata_location_name AS n
          ON n.qid = g.qid
        WHERE g.geonames_id = CAST(? AS TEXT)

        ORDER BY source, source_lang, name
    """, (geonames_id, geonames_id))

    return [
        NameCandidate(
            source=row.source,
            source_lang=row.source_lang or '',
            name=row.name,
            qid=row.qid,
        )
        for row in df.itertuples(index=False)
    ]

def compare_one_combined(title: str, geonames_id: int, mcn_name: str) -> CombinedComparisonResult:
    candidates = get_nte_names_for_geonames_id(geonames_id)

    if not candidates:
        return CombinedComparisonResult(title, geonames_id, mcn_name, (), (), (), MatchStatus.DB_EMPTY)

    exact = [c for c in candidates if c.name == mcn_name]
    if exact:
        return CombinedComparisonResult(
            title=title,
            geonames_id=geonames_id,
            mcn_name=mcn_name,
            matched_sources=tuple(sorted(set(c.source for c in exact))),
            matched_langs=_uniq(f"{c.source}:{c.source_lang}" for c in exact),
            matched_qids=_uniq(c.qid for c in exact),
            status=MatchStatus.EXACT,
        )

    folded = [c for c in candidates if c.name.lower() == mcn_name.lower()]
    if folded:
        return CombinedComparisonResult(
            title=title,
            geonames_id=geonames_id,
            mcn_name=mcn_name,
            matched_sources=tuple(sorted(set(c.source for c in folded))),
            matched_langs=_uniq(f"{c.source}:{c.source_lang}" for c in folded),
            matched_qids=_uniq(c.qid for c in folded),
            status=MatchStatus.CASE_FOLD,
        )

    return CombinedComparisonResult(title, geonames_id, mcn_name, (), (), (), MatchStatus.DB_MISS)

mcn_geonames_ids = sorted({gid for _, gid in ck3_latin_names})
wd_link_rows = []
for gid in mcn_geonames_ids:
    df = run_query("""
        SELECT qid
        FROM wikidata_location_geonames
        WHERE geonames_id = CAST(? AS TEXT)
        ORDER BY qid
    """, (gid,))
    if not df.empty:
        wd_link_rows.append((gid, df["qid"].tolist()))

print(f"MCN GeoNames IDs checked          : {len(mcn_geonames_ids)}")
print(f"  — with linked Wikidata QID(s)   : {len(wd_link_rows)} ({len(wd_link_rows)/len(mcn_geonames_ids):.1%})")
print(f"  — total linked Wikidata QID rows: {sum(len(qids) for _, qids in wd_link_rows)}")

combined_results = [
    compare_one_combined(title, gid, mcn_name)
    for (title, gid), mcn_name in ck3_latin_attested.items()
]

combined_counts = collections.Counter(r.status for r in combined_results)
print("\nMCN Latin attested names: GeoNames + Wikidata")
for status in MatchStatus:
    n = combined_counts[status]
    print(f"{status.value:15s}  {n:4d}  ({n/len(combined_results):.1%})")

baseline_counts = collections.Counter(r.status for r in results)
summary = pd.DataFrame([
    {
        "status": status.value,
        "geonames_only": baseline_counts[status],
        "geonames_plus_wikidata": combined_counts[status],
        "delta": combined_counts[status] - baseline_counts[status],
    }
    for status in MatchStatus
])
display(summary)

hit_statuses = {MatchStatus.EXACT, MatchStatus.CASE_FOLD}
geonames_hit_keys = {
    (r.title, r.geonames_id)
    for r in results
    if r.status in hit_statuses
}
combined_hit_keys = {
    (r.title, r.geonames_id)
    for r in combined_results
    if r.status in hit_statuses
}
added_by_wikidata = [
    r for r in combined_results
    if (r.title, r.geonames_id) in combined_hit_keys - geonames_hit_keys
]

source_buckets = collections.Counter(
    "+".join(r.matched_sources)
    for r in combined_results
    if r.status in hit_statuses
)
print("\nHit source buckets:")
for bucket, n in source_buckets.most_common():
    print(f"{bucket:20s}  {n:4d}")

print(f"\nAdditional MCN hits contributed by Wikidata: {len(added_by_wikidata)}")
if added_by_wikidata:
    display(pd.DataFrame([
        {
            "title": r.title,
            "geonames_id": r.geonames_id,
            "mcn_name": r.mcn_name,
            "status": r.status.value,
            "matched_sources": ", ".join(r.matched_sources),
            "matched_langs": ", ".join(r.matched_langs),
            "matched_qids": ", ".join(r.matched_qids),
        }
        for r in sorted(added_by_wikidata, key=lambda x: x.title)
    ]))

MCN GeoNames IDs checked          : 193
  — with linked Wikidata QID(s)   : 69 (35.8%)
  — total linked Wikidata QID rows: 69

MCN Latin attested names: GeoNames + Wikidata
exact              79  (66.9%)
case_fold           0  (0.0%)
db_empty            0  (0.0%)
db_miss            39  (33.1%)


,status,geonames_only,geonames_plus_wikidata,delta
0,exact,63,79,16
1,case_fold,0,0,0
2,db_empty,0,0,0
3,db_miss,55,39,-16



Hit source buckets:
geonames                50
wikidata                16
geonames+wikidata       13

Additional MCN hits contributed by Wikidata: 16


,title,geonames_id,mcn_name,status,matched_sources,matched_langs,matched_qids
0,c_aalborg,2624886,Alburgum,exact,wikidata,wikidata:la,Q25410
1,c_beloozero,577711,Bielozera,exact,wikidata,wikidata:la,Q104764
2,c_kassel,3220995,Cassala,exact,wikidata,wikidata:la,Q2865
3,c_kontupohja,545626,Condopoga,exact,wikidata,wikidata:la,Q155108
4,c_lubusz,6551072,Lebusium,exact,wikidata,wikidata:la,Q26362
5,c_mugodzhar_hills,608797,Norossus,exact,wikidata,wikidata:la,Q1364338
6,c_nizhny_novgorod,520555,Novogardia Inferior,exact,wikidata,wikidata:la,Q891
7,c_nyitra,3058531,Nitria,exact,wikidata,wikidata:la,Q26397
8,c_pavlodar,1520240,Paulodorum,exact,wikidata,wikidata:la,Q486282
9,c_rhodos,400666,Rhodus,exact,wikidata,wikidata:la,Q188731


In [9]:
# For titles MCN has no Latin name for, check what the DB does have

@dataclass
class MissingResult:
    title:       str
    geonames_id: int
    la_names:    list[str]
    untagged:    list[str]   # rows with empty isolanguage
    status:      str         # 'db_has_la' / 'db_has_untagged' / 'db_has_nothing'

def check_missing(title: str, geonames_id: int) -> MissingResult:
    df = run_query("""
        SELECT isolanguage, alternate_name
        FROM alternate_name
        WHERE geonameid = ?
          AND isolanguage IN ('la', '')
        ORDER BY isolanguage DESC, alternate_name
    """, (geonames_id,))

    la_names  = df.loc[df["isolanguage"] == "la",  "alternate_name"].tolist()
    untagged  = df.loc[df["isolanguage"] == "",     "alternate_name"].tolist()

    if la_names:
        status = "db_has_la"
    elif untagged:
        status = "db_has_untagged"
    else:
        status = "db_has_nothing"

    return MissingResult(title, geonames_id, la_names, untagged, status)

missing_results = [
    check_missing(title, gid)
    for (title, gid) in ck3_latin_missing
]

# Summary
miss_counts = collections.Counter(r.status for r in missing_results)
for status, n in miss_counts.most_common():
    print(f"{status:20s}  {n:4d}  ({n/len(missing_results):.1%})")

print()

db_has = [r for r in missing_results if r.status != "db_has_nothing"]
for r in sorted(db_has, key=lambda r: r.title):
    names = r.la_names or r.untagged
    tag   = "la" if r.la_names else "untagged"
    print(f"{r.title:35s}  [{tag}]  {names}")

db_has_untagged         75  (75.0%)
db_has_nothing          17  (17.0%)
db_has_la                8  (8.0%)

c_SUM_pekanbaru                      [untagged]  ['Pakanbahru', 'Pakanbaroe', 'Pakanbaru']
c_abomey                             [untagged]  ['Abome', 'Abomey']
c_al-gharbiya                        [untagged]  ['Al Gharbīyah']
c_al_aqabah                          [untagged]  ['Qal`at el `Aqaba', '`Aqaba']
c_aqtobe                             [untagged]  ['Aktiubinsk', 'Aktobe', 'Aktubinsk', 'Aktyubinsk', 'Ukhtiubinskii']
c_atkarsk                            [untagged]  ['Atarsk', 'Atkarsk']
c_baydhabo                           [untagged]  ['Baidabho', 'Baidabo', 'Iscia Baidoa', 'Isha Baydabo', 'Isha Baydhaba', 'Isha Baydhabo']
c_busaso                             [untagged]  ['Bandar Kassim', 'Bender Qaasin', 'Boosaso', 'Bosaso', 'Bosasso', 'Bénder Qāsin']
c_buzachi                            [untagged]  ['Bozashchy Tübegi', 'Buzachi Peninsula', 'Halbinsel Busatschi', 'Poluostrov 

In [10]:
@dataclass
class CombinedMissingResult:
    title:              str
    geonames_id:        int
    geonames_la:        list[str]
    wikidata_la:        list[str]
    geonames_untagged:  list[str]
    wikidata_qids:      list[str]
    status:             str

def check_missing_combined(title: str, geonames_id: int) -> CombinedMissingResult:
    geonames_df = run_query("""
        SELECT isolanguage, alternate_name
        FROM alternate_name
        WHERE geonameid = ?
          AND isolanguage IN ('la', '')
        ORDER BY isolanguage DESC, alternate_name
    """, (geonames_id,))

    wikidata_df = run_query("""
        SELECT DISTINCT n.qid, n.name
        FROM wikidata_location_geonames AS g
        JOIN wikidata_location_name AS n
          ON n.qid = g.qid
        WHERE g.geonames_id = CAST(? AS TEXT)
          AND (n.geo_lang = 'la' OR n.wd_lang = 'la')
        ORDER BY n.qid, n.name
    """, (geonames_id,))

    geonames_la = geonames_df.loc[
        geonames_df["isolanguage"] == "la",
        "alternate_name",
    ].tolist()
    geonames_untagged = geonames_df.loc[
        geonames_df["isolanguage"] == "",
        "alternate_name",
    ].tolist()
    wikidata_la = wikidata_df["name"].tolist() if not wikidata_df.empty else []
    wikidata_qids = wikidata_df["qid"].drop_duplicates().tolist() if not wikidata_df.empty else []

    if geonames_la and wikidata_la:
        status = "db_has_la_both"
    elif geonames_la:
        status = "db_has_la_geonames"
    elif wikidata_la:
        status = "db_has_la_wikidata"
    elif geonames_untagged:
        status = "db_has_geonames_untagged"
    else:
        status = "db_has_nothing"

    return CombinedMissingResult(
        title=title,
        geonames_id=geonames_id,
        geonames_la=geonames_la,
        wikidata_la=wikidata_la,
        geonames_untagged=geonames_untagged,
        wikidata_qids=wikidata_qids,
        status=status,
    )

combined_missing_results = [
    check_missing_combined(title, gid)
    for (title, gid) in ck3_latin_missing
]

combined_miss_counts = collections.Counter(r.status for r in combined_missing_results)
for status, n in combined_miss_counts.most_common():
    print(f"{status:28s}  {n:4d}  ({n/len(combined_missing_results):.1%})")

combined_missing_df = pd.DataFrame([
    {
        "title": r.title,
        "geonames_id": r.geonames_id,
        "status": r.status,
        "geonames_la": r.geonames_la,
        "wikidata_la": r.wikidata_la,
        "geonames_untagged": r.geonames_untagged,
        "wikidata_qids": r.wikidata_qids,
    }
    for r in sorted(combined_missing_results, key=lambda x: x.title)
    if r.status != "db_has_nothing"
])

display(combined_missing_df)

db_has_geonames_untagged        68  (68.0%)
db_has_nothing                  16  (16.0%)
db_has_la_wikidata               8  (8.0%)
db_has_la_geonames               8  (8.0%)


,title,geonames_id,status,geonames_la,wikidata_la,geonames_untagged,wikidata_qids
0,c_SUM_pekanbaru,1631761,db_has_geonames_untagged,[],[],"[Pakanbahru, Pakanbaroe, Pakanbaru]",[]
1,c_abomey,2395915,db_has_geonames_untagged,[],[],"[Abome, Abomey]",[]
2,c_al-gharbiya,7753427,db_has_geonames_untagged,[],[],[Al Gharbīyah],[]
3,c_al_aqabah,250774,db_has_la_wikidata,[],[Aelana],"[Qal`at el `Aqaba, `Aqaba]",[Q180522]
4,c_aqtobe,610611,db_has_la_wikidata,[],[Aktobe],"[Aktiubinsk, Aktobe, Aktubinsk, Aktyubinsk, Ukhtiubinskii]",[Q477232]
5,c_atkarsk,580420,db_has_geonames_untagged,[],[],"[Atarsk, Atkarsk]",[]
6,c_baydhabo,64536,db_has_geonames_untagged,[],[],"[Baidabho, Baidabo, Iscia Baidoa, Isha Baydabo, Isha Baydhaba, Isha Baydhabo]",[]
7,c_busaso,64013,db_has_geonames_untagged,[],[],"[Bandar Kassim, Bender Qaasin, Boosaso, Bosaso, Bosasso, Bénder Qāsin]",[]
8,c_buzachi,610203,db_has_geonames_untagged,[],[],"[Bozashchy Tübegi, Buzachi Peninsula, Halbinsel Busatschi, Poluostrov Buzachi]",[]
9,c_canavese,8978041,db_has_geonames_untagged,[],[],[Canavese],[]


In [11]:
combined_misses = [
    r for r in combined_results
    if r.status == MatchStatus.DB_MISS
]

combined_miss_df = pd.DataFrame([
    {
        "title": r.title,
        "geonames_id": r.geonames_id,
        "mcn_name": r.mcn_name,
        "status": r.status.value,
    }
    for r in sorted(combined_misses, key=lambda x: (x.mcn_name, x.title))
])

print(f"MCN-attested Latin names absent from NTE: {len(combined_miss_df)}")
display(combined_miss_df)

MCN-attested Latin names absent from NTE: 39


,title,geonames_id,mcn_name,status
0,d_talas_alatau,1490299,Alatau Talassicus,db_miss
1,c_arnsberg,6557667,Arnsburgum,db_miss
2,c_badakhshan,1147745,Badachsania,db_miss
3,d_badakhshan,1147745,Badachsania,db_miss
4,c_medjerda,2488775,Bagrada,db_miss
5,c_campulung,665024,Berzovia,db_miss
6,c_brzeg,3102459,Brega,db_miss
7,c_euchaita,750638,Carissa,db_miss
8,c_dnipro,565896,Danaper,db_miss
9,c_debar,791606,Deborus,db_miss
